# W5D4_W5D5 — Dropout, Overfitting and Activation Functions

في التاسك ده كملت على MNIST، وجربت Dropout وشفت تأثيره على الـ Validation Loss، وبعدها قارنت ReLU مع Sigmoid. في الآخر كتبت ملخص بسيط عن CNN وRNN وTransformer.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

transform = transforms.ToTensor()
train_full = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_size = 50000
val_size = 10000
train_dataset, val_dataset = random_split(train_full, [train_size, val_size], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

print('Training samples:', len(train_dataset))
print('Validation samples:', len(val_dataset))
print('Testing samples:', len(test_dataset))

In [ ]:
class MNISTNet(nn.Module):
    def __init__(self, activation='relu', dropout=0.0):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28 * 28, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)
        self.dropout = nn.Dropout(p=dropout)
        self.activation = nn.ReLU() if activation == 'relu' else nn.Sigmoid()

    def forward(self, x):
        x = self.flatten(x)
        x = self.activation(self.fc1(x))
        x = self.dropout(x)
        x = self.activation(self.fc2(x))
        x = self.dropout(x)
        return self.fc3(x)

criterion = nn.CrossEntropyLoss()

def run_experiment(activation='relu', dropout=0.0, epochs=5):
    model = MNISTNet(activation=activation, dropout=dropout).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * images.size(0)
            train_correct += (outputs.argmax(1) == labels).sum().item()
            train_total += labels.size(0)

        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * images.size(0)
                val_correct += (outputs.argmax(1) == labels).sum().item()
                val_total += labels.size(0)

        train_loss /= train_total
        val_loss /= val_total
        train_acc = train_correct / train_total
        val_acc = val_correct / val_total

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        print(f'Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Train Acc: {train_acc:.2%} | Val Acc: {val_acc:.2%}')

    return model, history

## 1) Dropout مقابل الموديل العادي

هنا بستخدم نفس الفكرة الأساسية للموديل، لكن مرة من غير Dropout ومرة بـ `Dropout(p=0.25)`.

In [ ]:
baseline_model, baseline_history = run_experiment(activation='relu', dropout=0.0, epochs=5)
dropout_model, dropout_history = run_experiment(activation='relu', dropout=0.25, epochs=5)

In [ ]:
epochs = range(1, 6)

plt.figure(figsize=(8, 5))
plt.plot(epochs, baseline_history['val_loss'], marker='o', label='Without Dropout')
plt.plot(epochs, dropout_history['val_loss'], marker='o', label='Dropout p=0.25')
plt.xlabel('Epoch')
plt.ylabel('Validation Loss')
plt.title('Validation Loss: Dropout vs No Dropout')
plt.legend()
plt.grid(True)
plt.show()

print('Final validation loss without Dropout:', round(baseline_history['val_loss'][-1], 4))
print('Final validation loss with Dropout:', round(dropout_history['val_loss'][-1], 4))

### ملاحظتي

الـ Dropout بيوقف شوية neurons بشكل عشوائي أثناء التدريب، فالموديل مايعتمدش زيادة عن اللزوم على neurons معينة. الفكرة الأساسية هي تقليل الـ Overfitting وتحسين الـ Generalization. النتيجة الفعلية للمقارنة موجودة في الـ plot والـ values اللي بيطلعها الكود، لأن النتائج ممكن تختلف شوية حسب الـ device والـ run.

## 2) ReLU مقابل Sigmoid

هنا غيرت الـ Activation Function بس، وخليت باقي الإعدادات زي ما هي قدر الإمكان عشان المقارنة تبقى أوضح.

In [ ]:
relu_model, relu_history = run_experiment(activation='relu', dropout=0.25, epochs=5)
sigmoid_model, sigmoid_history = run_experiment(activation='sigmoid', dropout=0.25, epochs=5)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(epochs, relu_history['val_loss'], marker='o', label='ReLU')
plt.plot(epochs, sigmoid_history['val_loss'], marker='o', label='Sigmoid')
plt.xlabel('Epoch')
plt.ylabel('Validation Loss')
plt.title('ReLU vs Sigmoid')
plt.legend()
plt.grid(True)
plt.show()

print('Final ReLU validation loss:', round(relu_history['val_loss'][-1], 4))
print('Final Sigmoid validation loss:', round(sigmoid_history['val_loss'][-1], 4))

## 3) الخلاصة السريعة

- **ReLU:** عادةً بتتعلم أسرع في الشبكات العميقة وبتساعد في تقليل مشكلة الـ vanishing gradients مقارنةً بـ Sigmoid.
- **Sigmoid:** بسيطة ومفيدة في حالات معينة، لكن ممكن التعلم يبقى أبطأ في الشبكات العميقة بسبب الـ saturation والـ vanishing gradients.
- **Dropout:** وسيلة Regularization بتساعد تقلل الـ Overfitting بدل ما الموديل يحفظ بيانات التدريب بشكل زيادة.